# 1.2 Intro to Mech Interp — Part 1: TransformerLens, the cache, and attention by hand

ARENA 1.2 has 4 parts: (1) TransformerLens intro, (2) finding induction heads, (3) hooks, (4) reverse-engineering induction circuits.
This notebook is part 1.

**Library note.** ARENA's code uses `HookedTransformer`, which TransformerLens 4.x removed. We load with
`TransformerBridge.boot_transformers("gpt2")` + `.enable_compatibility_mode()`. Compatibility mode applies the same
weight processing as old `HookedTransformer.from_pretrained` (fold LayerNorm, centre writing weights and unembed),
so numbers match ARENA's answer key. Everything after loading (`run_with_cache`, `cache[...]`, `W_Q`...) is the same.

**Convention.** Code stores tokens as rows: activations are `[batch, pos, d_model]`, weights are `[d_in, d_out]`
(e.g. `W_Q[layer]` is `[n_heads, d_model, d_head]`, used as `x @ W_Q`). Your notes use columns and `[out, in]`
(`q = W_Q x`). Same maps, transposed.

In [ ]:
# --- Setup (boilerplate) ---
import importlib.metadata as md
import torch
from transformer_lens.model_bridge import TransformerBridge

torch.manual_seed(0)
torch.set_grad_enabled(False)          # we only read activations; no backward pass needed
device = "cuda" if torch.cuda.is_available() else "cpu"

model = TransformerBridge.boot_transformers("gpt2", device=device)   # fp32 by default
model.enable_compatibility_mode()      # HookedTransformer-equivalent weights (fold LN, centre W_out, W_U)

for pkg in ["torch", "transformer_lens", "transformers"]:
    print(f"{pkg:18} {md.version(pkg)}")   # copy these into README.md
print("device:", device, "| dtype:", model.cfg.dtype)

## 1. The config

**Predict before running:** for GPT-2 Small, write down `n_layers`, `n_heads`, `d_model`, `d_head`, `d_mlp`, `d_vocab`, `n_ctx`.
Then check. One relation between them should hold exactly; which one, and why does the architecture need it?

In [ ]:
cfg = model.cfg
for name in ["n_layers", "n_heads", "d_model", "d_head", "d_mlp", "d_vocab", "n_ctx", "normalization_type", "positional_embedding_type"]:
    print(f"{name:28} {getattr(cfg, name, 'n/a')}")

## 2. One forward pass and the cache

Traps to watch: `to_tokens` **prepends BOS** (`<|endoftext|>`, id 50256) by default, so position 0 is BOS, not your first word.
GPT-2 tokens usually carry the leading space (`" Paris"` ≠ `"Paris"`).

In [ ]:
prompt = "Mechanistic interpretability is the study of how neural networks compute."
tokens = model.to_tokens(prompt)               # [batch, pos] = [1, n_pos], BOS at pos 0
logits, cache = model.run_with_cache(tokens)   # logits [batch, pos, d_vocab] = [1, n_pos, 50257]

n_pos = tokens.shape[1]
print("n_pos:", n_pos)
print(model.to_str_tokens(tokens))            # see the BOS and the leading spaces

## 3. Exercise (you): the shape table

Fill `predicted` with the shape you expect for each cache entry, **before** running the check cell.
Use names, not just numbers, in your head: e.g. `[batch, pos, head, d_head]`.

Questions to answer in a markdown cell below:
- Why is `pattern` 4-D but `q` is also 4-D with a different axis order? Which axis is summed over when attention is applied?
- `mlp_out` and `attn_out` have the same shape as `resid_pre`. Why must they?
- Which entries live in the residual stream (`d_model`) and which live inside a head or the MLP?

In [ ]:
# YOUR PREDICTIONS: replace each None with a tuple, e.g. (1, n_pos, 768)
L = 0   # layer to inspect
predicted = {
    "embed":              None,   # cache["embed"]           token embedding W_E[tokens]
    "pos_embed":          None,   # cache["pos_embed"]       positional embedding W_pos[pos]
    ("resid_pre", L):     None,   # residual stream entering block L
    ("q", L):             None,   # queries
    ("k", L):             None,   # keys
    ("v", L):             None,   # values
    ("attn_scores", L):   None,   # q·k / sqrt(d_head), masked, before softmax
    ("pattern", L):       None,   # softmax of scores
    ("z", L):             None,   # pattern-weighted values, per head
    ("attn_out", L):      None,   # sum over heads of z @ W_O, plus b_O
    ("resid_mid", L):     None,   # resid_pre + attn_out
    ("pre", L):           None,   # MLP hidden pre-activation
    ("post", L):          None,   # MLP hidden after GELU
    ("mlp_out", L):       None,   # MLP output
    ("resid_post", L):    None,   # resid_mid + mlp_out
}

In [ ]:
# --- Check (boilerplate): compares your table with the cache ---
for key, pred in predicted.items():
    try:
        actual = tuple(cache[key].shape)
    except Exception as e:
        print(f"{str(key):26} MISSING in cache ({type(e).__name__}) -> tell me, may be a TL 4.x alias gap")
        continue
    mark = "ok " if pred == actual else "XX "
    print(f"{mark} {str(key):26} predicted {pred}  actual {actual}")

## 4. Exercise (you): recompute layer 0's attention pattern by hand

**Question.** Does `cache["pattern", 0]` equal softmax of masked, scaled `q·k`, computed from the cached `q` and `k`?

**Metric.** `(mine - cache["pattern", 0]).abs().max()`.

**Expected.** Below ~1e-5 in fp32 (only float rounding differs). Each row of the pattern sums to 1, and the upper
triangle (k_pos > q_pos) is exactly 0.

**Inputs:** `cache["q", 0]`, `cache["k", 0]` (shapes from your table), `model.cfg.d_head`.
**Output:** `[batch, head, q_pos, k_pos]` — note the axis order differs from `q`/`k`.

Steps to think through, not code to copy:
1. Scores: for each head, dot every query position with every key position. Which axes do you contract, and which do you keep?
2. Scale: divide by what, and why? (Hint: variance of a dot product of `d_head` unit-variance terms.)
3. Causal mask: which entries, and why set them to `-inf` rather than 0 **before** softmax?
4. Softmax over which axis?

**Predict before running:** what happens to the max error if you (a) skip the scale, (b) mask with 0 instead of -inf,
(c) softmax over the wrong axis? Write numbers or "large/small" first, then try each.

In [ ]:
# YOUR CODE: compute my_pattern  [batch, head, q_pos, k_pos]

In [ ]:
# --- Check (boilerplate) ---
ref = cache["pattern", 0]                                   # [batch, head, q_pos, k_pos] = [1, 12, n_pos, n_pos]
assert my_pattern.shape == ref.shape, (my_pattern.shape, ref.shape)
print("max abs error:", (my_pattern - ref).abs().max().item())
print("rows sum to 1:", torch.allclose(my_pattern.sum(-1), torch.ones_like(my_pattern.sum(-1))))
print("upper triangle zero:", (my_pattern.triu(diagonal=1) == 0).all().item())